# Plotly Dashboard — Enhanced

This notebook contains the cleaned & enhanced Plotly Dash dashboard for your cricket analytics project. Run the final code cell to start the dashboard locally (it will print the URL and open at `http://127.0.0.1:8050`).

Make sure project files are accessible (set `AI_PROJECT_ROOT` environment variable if needed).

In [3]:
"""
Cleaned and Enhanced Plotly Dash Dashboard for Cricket Analytics Project
- Reads data from the project's data/ folder and model(s) from models/ folder
- Robust: safe loading, column normalization, single callback, error handling
- Requires: dash, dash-bootstrap-components, pandas, plotly, joblib (for models)
"""

import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import joblib
import warnings

warnings.filterwarnings("ignore")

# ---------- Configuration ----------
ROOT = Path(os.getcwd())
# If running from a different directory, allow using environment variable PROJECT_ROOT
PROJECT_ROOT = Path(os.environ.get("AI_PROJECT_ROOT", r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project"))
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# Expected files (fallbacks allowed)
FILES = {
    "final": DATA_DIR / "final_processed_data.csv",
    "batsman_profiles": DATA_DIR / "batsman_profiles.csv",
    "batsman_similarity": DATA_DIR / "batsman_similarity.csv",
    "bowler_stats": DATA_DIR / "bowler_stats.csv",
    "bowling_success": DATA_DIR / "bowling_success_model.csv",
    "phase_sr": DATA_DIR / "phase_sr.csv",
}


# ---------- Utilities ----------
def safe_csv_load(path: Path):
    if not path or not path.exists():
        print(f"[warn] file not found: {path}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        print(f"[info] Loaded {path} -> {df.shape}")
        return df
    except Exception as e:
        print(f"[warn] Failed loading {path}: {e}")
        return pd.DataFrame()


def safe_joblib_load(path: Path):
    if not path or not path.exists():
        print(f"[warn] model not found: {path}")
        return None
    try:
        model = joblib.load(path)
        print(f"[info] Loaded model {path}")
        return model
    except Exception as e:
        print(f"[warn] Failed loading model {path}: {e}")
        return None


# ---------- Feature-engineered datasets for each trained model ----------
FEATURE_DATA_DIR = PROJECT_ROOT / "data" / "model_feature_engineered_datasets"

MODEL_DATASETS = {
    "good_ball": safe_csv_load(FEATURE_DATA_DIR / "df_model_good_ball.csv"),
    "ball_type": safe_csv_load(FEATURE_DATA_DIR / "df_model_ball_type.csv"),
    "boundary": safe_csv_load(FEATURE_DATA_DIR / "df_model_boundary.csv"),
    "line": safe_csv_load(FEATURE_DATA_DIR / "df_model_line.csv"),
}


# ---------- Load datasets & models ----------
df_final = safe_csv_load(FILES["final"])
batsman_profiles = safe_csv_load(FILES["batsman_profiles"])
batsman_similarity = pd.read_csv(FILES["batsman_similarity"], index_col=0)
batsman_similarity.index = batsman_similarity.index.astype(str).str.strip().str.title()
batsman_similarity.columns = batsman_similarity.columns.astype(str).str.strip().str.title()

bowler_stats = safe_csv_load(FILES["bowler_stats"])
bowling_success = safe_csv_load(FILES["bowling_success"])
phase_sr = safe_csv_load(FILES["phase_sr"])

batsman_similarity.columns = batsman_similarity.columns.str.strip().str.title()
df_final['Batsman_Name'] = df_final['Batsman_Name'].str.strip().str.title()

# load models if present (joblib)
predict_good_ball_model = safe_joblib_load(MODELS_DIR / "predict_good_ball_model_random_forest.joblib")


# ---------- Normalize column names and basic fixes ----------
def normalize_columns(df):
    if df is None or df.empty:
        return df
    df = df.copy()
    # common mappings
    colmap = {}
    if "Batsman_Name_clean" in df.columns and "Batsman_Name" not in df.columns:
        colmap["Batsman_Name_clean"] = "Batsman_Name"
    if "run" in df.columns and "Runs" not in df.columns:
        colmap["run"] = "Runs"
    if "Bowler_Name" in df.columns and "Bowler" not in df.columns:
        colmap["Bowler_Name"] = "Bowler"
    df.rename(columns=colmap, inplace=True)
    return df


for name in ["df_final", "batsman_profiles", "bowler_stats", "phase_sr"]:
    df = globals().get(name)
    globals()[name] = normalize_columns(df)

# Ensure season column is string
for df in [df_final, batsman_profiles, bowler_stats, phase_sr]:
    if df is not None and not df.empty and "season" in df.columns:
        df["season"] = df["season"].astype(str)


# ---------- Dash app ----------
app = Dash(__name__, external_stylesheets=[dbc.themes.LUMEN], suppress_callback_exceptions=True)
# Prevent duplicate callback registration in notebook environments
app._callbacks = {}
app.title = "Cricket Analytics — Enhanced Dashboard"
server = app.server


# ---------- Helper plotting functions ----------
def make_overview_cards(df):
    if df is None or df.empty:
        return dbc.Row([dbc.Col(html.Div("No data loaded"))])
    matches = int(df['match_obj_id'].nunique()) if 'match_obj_id' in df.columns else (int(df['match_id'].nunique()) if 'match_id' in df.columns else "—")
    players = int(pd.concat([df.get('Batsman_Name', pd.Series(dtype=str)), df.get('Bowler', pd.Series(dtype=str))]).nunique())
    seasons = df['season'].nunique() if 'season' in df.columns else "—"
    total_runs = int(df['Runs'].sum()) if 'Runs' in df.columns else "—"
    cards = dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Matches"), html.H3(matches)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Unique players"), html.H3(players)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Seasons"), html.H3(seasons)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Total runs"), html.H3(total_runs)])), md=3),
    ], class_name='mb-3')
    return cards


def batsman_heatmap(df, batsman, bins_line=12, bins_length=12):
    if df is None or df.empty:
        return go.Figure(layout=go.Layout(title="No data"))
    if 'pitchLine' not in df.columns or 'pitchLength' not in df.columns:
        if 'x' in df.columns and 'y' in df.columns:
            sub = df[df.get('Batsman_Name') == batsman] if batsman else df.sample(min(2000, len(df)))
            return px.density_heatmap(sub, x='x', y='y', nbinsx=bins_line, nbinsy=bins_length, title=f"Shot density — {batsman}")
        return go.Figure(layout=go.Layout(title="Pitch data not present"))
    sub = df[df.get('Batsman_Name') == batsman] if batsman else df.sample(min(2000, len(df)))
    if sub.empty:
        sub = df.sample(min(2000, len(df)))
    fig = px.density_heatmap(
        sub, x='pitchLine', y='pitchLength',
        nbinsx=bins_line, nbinsy=bins_length,
        labels={'pitchLine': 'Pitch line', 'pitchLength': 'Pitch length'},
        title=f"Pitch Map — {batsman or 'Overall'}"
    )
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    return fig


def phase_strike_rate(df, batsman=None):
    if df is None or df.empty:
        return go.Figure(layout=go.Layout(title="No phase SR data"))

    # Standardize column names
    cols = {c.lower(): c for c in df.columns}
    phase_col = cols.get('match_phase') or cols.get('phase')
    sr_col = cols.get('strike_rate')

    if not phase_col or not sr_col:
        return go.Figure(layout=go.Layout(title="Phase or Strike Rate column missing"))

    # Filter for batsman if available
    d = df[df.get('Batsman_Name') == batsman] if batsman and 'Batsman_Name' in df.columns else df
    if d.empty:
        return go.Figure(layout=go.Layout(title=f"No SR data for {batsman}"))

    # Average strike rate by phase
    fig = px.bar(
        d.groupby(phase_col, as_index=False)[sr_col].mean(),
        x=phase_col,
        y=sr_col,
        color=phase_col,
        title=f"Phase-wise Strike Rate — {batsman or 'Overall'}"
    )
    fig.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=13),
        margin=dict(l=20, r=20, t=40, b=20),
        showlegend=False
    )
    return fig


def top_similar_batsmen(sim_df, batsman, top_n=5):
    if sim_df is None or sim_df.empty:
        return go.Figure(layout=go.Layout(title="No similarity data"))

    # Ensure clean names
    sim_df.index = sim_df.index.astype(str).str.strip().str.title()
    sim_df.columns = sim_df.columns.astype(str).str.strip().str.title()

    # --- Smart mapping for short vs full names ---
    lookup_name = batsman
    if batsman not in sim_df.index:
        surname = batsman.split()[-1].lower()
        # Find closest full name using surname
        possible = [name for name in sim_df.index if name.lower().endswith(surname)]
        if possible:
            lookup_name = possible[0]

    # If still not found, exit gracefully
    if lookup_name not in sim_df.index:
        return go.Figure(layout=go.Layout(title=f"{batsman} not found in similarity data"))

    # Extract that batsman's similarity row
    s = sim_df.loc[lookup_name].dropna()

    # Remove self-similarity (score == 1.0)
    s = s[s.index != lookup_name]

    # Sort descending and pick top N
    s = s.sort_values(ascending=False).head(top_n)

    if s.empty:
        return go.Figure(layout=go.Layout(title=f"No similar batsmen found for {batsman}"))

    # Prepare DataFrame for plotting
    df = pd.DataFrame({
        "similar_batsman": s.index,
        "similarity_score": s.values
    })
    df["similarity_score"] = df["similarity_score"].round(3)

    # Plot
    fig = px.bar(
        df,
        x="similar_batsman",
        y="similarity_score",
        title=f"Top {top_n} similar batsmen to {lookup_name}",
        color="similarity_score",
        color_continuous_scale="Blues"
    )

    fig.update_layout(
        template="plotly_white",
        margin=dict(l=20, r=20, t=40, b=20),
        font=dict(family="Arial", size=13),
        showlegend=False
    )

    return fig


def top_bowlers_by_success(df_bowler, top_n=10):
    if df_bowler is None or df_bowler.empty:
        return go.Figure(layout=go.Layout(title="No bowler data"))
    si_col = next((c for c in df_bowler.columns if 'success' in c.lower()), None)
    if si_col is None:
        # try wickets or economy as fallback
        if 'totalWickets' in df_bowler.columns:
            si_col = 'totalWickets'
        else:
            numeric = df_bowler.select_dtypes(include='number').columns.tolist()
            si_col = numeric[0] if numeric else None
    if si_col is None:
        return go.Figure(layout=go.Layout(title="No metric columns in bowler stats"))
    d = df_bowler.sort_values(si_col, ascending=False).head(top_n)
    xcol = 'Bowler_Name' if 'Bowler_Name' in d.columns else (d.columns[0] if len(d.columns) > 0 else None)
    if xcol is None:
        return go.Figure(layout=go.Layout(title="No bowler name column"))
    fig = px.bar(d, x=xcol, y=si_col, title='Top bowlers by ' + si_col)
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    return fig


# ---------- Layout ----------
batsmen = sorted(df_final['Batsman_Name'].dropna().unique()) if not df_final.empty and 'Batsman_Name' in df_final.columns else []
batsman_options = [{'label': b, 'value': b} for b in batsmen]
batsman_options.insert(0, {'label': 'Overall', 'value': 'Overall'})

app.layout = dbc.Container([
    html.H2("Cricket Analytics — Strategy Dashboard", style={'marginTop': 10}),
    html.Hr(),
    make_overview_cards(df_final),
    dbc.Card([dbc.CardBody([
        dbc.Row([
            dbc.Col([html.Label('Batsman'), dcc.Dropdown(id='batsman-filter', options=batsman_options, value=batsmen[0] if batsmen else None)], md=4),
            dbc.Col([html.Label('Season'), dcc.Dropdown(id='season-filter', options=[{'label': s, 'value': s} for s in sorted(df_final['season'].dropna().unique())] if 'season' in df_final.columns else [], multi=True)], md=4),
            dbc.Col([html.Label('Team'), dcc.Dropdown(id='team-filter', options=[{'label': t, 'value': t} for t in sorted(df_final['Batting_Team'].dropna().unique())] if 'Batting_Team' in df_final.columns else [])], md=4),
        ])
    ])], class_name='mb-3'),
    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Batsman: Heatmap'), dbc.CardBody(dcc.Graph(id='heatmap-graph'))]), md=6),
        dbc.Col(dbc.Card([dbc.CardHeader('Phase-wise Strike Rate'), dbc.CardBody(dcc.Graph(id='phase-sr-graph'))]), md=6),
    ], class_name='mb-3'),
    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Top similar batsmen'), dbc.CardBody(dcc.Graph(id='similar-batsmen-graph'))]), md=6),
        dbc.Col(dbc.Card([dbc.CardHeader('Top bowlers by success'), dbc.CardBody(dcc.Graph(id='top-bowlers-graph'))]), md=6),
    ], class_name='mb-3'),
    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Filtered rows (preview)'), dbc.CardBody(html.Div(id='table-preview'))]), md=8),
        dbc.Col(dbc.Card([dbc.CardHeader('Model predictions'), dbc.CardBody(html.Div(id='model-predictions'))]), md=4),
    ], class_name='mb-3')
], fluid=True)


# ---------- Callbacks ----------
@app.callback(
    Output("heatmap-graph", "figure"),
    Output("phase-sr-graph", "figure"),
    Output("similar-batsmen-graph", "figure"),
    Output("top-bowlers-graph", "figure"),
    Output("table-preview", "children"),
    Output("model-predictions", "children"),
    Input("batsman-filter", "value"),
    Input("season-filter", "value"),
    Input("team-filter", "value"),
)
def update_dashboard(batsman, seasons, team):
    if batsman in [None, '', 'Overall']:
        batsman_val = df_final['Batsman_Name'].dropna().iloc[0] if 'Batsman_Name' in df_final.columns and not df_final.empty else None
    else:
        batsman_val = batsman

    # Filter data safely
    d = df_final.copy() if not df_final.empty else pd.DataFrame()
    if seasons:
        d = d[d['season'].isin(seasons)]
    if team:
        d = d[d['Batting_Team'] == team]

    batsman_val = batsman if batsman else None

    try:
        heat = batsman_heatmap(d if not d.empty else df_final, batsman_val)
    except Exception as e:
        print('Heatmap error:', e)
        heat = go.Figure()

    try:
        phase = phase_strike_rate(phase_sr, batsman_val)
    except Exception as e:
        print('Phase SR error:', e)
        phase = go.Figure()

    try:
        sim = top_similar_batsmen(batsman_similarity, batsman_val)
    except Exception as e:
        print('Similarity error:', e)
        sim = go.Figure()

    try:
        bowl = top_bowlers_by_success(bowling_success)
    except Exception as e:
        print('Bowlers error:', e)
        bowl = go.Figure()

    table_html = html.Div([html.H6(f"Showing {min(len(d), 200)} rows"),
                           d.head(25).to_html(classes='table table-striped', index=False)]) if not d.empty else html.Div("No rows for selected filters.")

    # Model predictions using correct engineered dataset
        # Model predictions using correct engineered dataset
        # Model predictions using correct engineered dataset
        # ---------- Model Predictions Section ----------
    model_div = html.Div("No model available")

    if predict_good_ball_model is not None:
        try:
            df_engineered = MODEL_DATASETS["good_ball"].copy()

            # --- Filter data based on dashboard selections ---
            if batsman and batsman not in [None, '', 'Overall'] and 'Batsman_Name' in df_engineered.columns:
                df_engineered = df_engineered[df_engineered['Batsman_Name'].str.contains(batsman, case=False, na=False)]
            if team and 'Batting_Team' in df_engineered.columns:
                df_engineered = df_engineered[df_engineered['Batting_Team'] == team]
            if seasons and 'season' in df_engineered.columns:
                df_engineered = df_engineered[df_engineered['season'].isin(seasons)]

            # --- If filtered empty, skip gracefully ---
            if df_engineered.empty:
                model_div = html.Div("No data available for this selection.")
            else:
                # Ensure required columns exist
                required_features = list(predict_good_ball_model.feature_names_in_) \
                    if hasattr(predict_good_ball_model, "feature_names_in_") else df_engineered.select_dtypes(include='number').columns.tolist()

                for col in required_features:
                    if col not in df_engineered.columns:
                        df_engineered[col] = 0.0

                # Encode categorical columns
                df_encoded = df_engineered.copy()
                for col in df_encoded.select_dtypes(include='object').columns:
                    df_encoded[col] = df_encoded[col].astype('category').cat.codes

                X = df_encoded.reindex(columns=required_features, fill_value=0)

                # --- Run Predictions ---
                if hasattr(predict_good_ball_model, 'predict_proba'):
                    df_engineered['good_ball_prob'] = predict_good_ball_model.predict_proba(X)[:, 1]
                else:
                    df_engineered['good_ball_prob'] = predict_good_ball_model.predict(X)

                # --- Create Summary Stats ---
                avg_prob = df_engineered['good_ball_prob'].mean()
                top10 = df_engineered.nlargest(10, 'good_ball_prob')

                # --- Visualizations ---
                prob_chart = px.histogram(
                    df_engineered,
                    x='good_ball_prob',
                    nbins=20,
                    title=f"Distribution of Good Ball Probabilities — {batsman or 'All Batsmen'}",
                    color_discrete_sequence=['#5DADE2']
                )

                donut_chart = go.Figure(
                    go.Indicator(
                        mode="gauge+number",
                        value=avg_prob * 100,
                        title={'text': f"Avg Good Ball % — {batsman or 'Overall'}"},
                        gauge={'axis': {'range': [0, 100]},
                               'bar': {'color': "#1F77B4"},
                               'steps': [{'range': [0, 50], 'color': "#F5B7B1"},
                                         {'range': [50, 100], 'color': "#ABEBC6"}]}
                    )
                )

                top_chart = px.bar(
                    top10,
                    y='good_ball_prob',
                    title=f"Top 10 Balls with Highest Good-Ball Probability — {batsman or 'All Batsmen'}",
                    color='good_ball_prob',
                    color_continuous_scale='Blues'
                )

                # --- Combine Results ---
                model_div = html.Div([
                    html.H6(f"Model Prediction Summary — {batsman or 'Overall'}"),
                    html.P(f"Average Good Ball Probability: {avg_prob:.2f}"),
                    dbc.Row([
                        dbc.Col(dcc.Graph(figure=donut_chart), md=4),
                        dbc.Col(dcc.Graph(figure=prob_chart), md=8),
                    ]),
                    html.Hr(),
                    dcc.Graph(figure=top_chart)
                ])

        except Exception as e:
            import traceback
            print("Model prediction error:", e)
            print(traceback.format_exc())
            model_div = html.Div("Model error: see server logs")




    return heat, phase, sim, bowl, table_html, model_div


# ---------- Run ----------
if __name__ == '__main__':
    print('Starting enhanced dashboard at http://127.0.0.1:8050')
    app.run(debug=True)


[warn] file not found: C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\model_feature_engineered_datasets\df_model_good_ball.csv
[warn] file not found: C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\model_feature_engineered_datasets\df_model_ball_type.csv
[warn] file not found: C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\model_feature_engineered_datasets\df_model_boundary.csv
[warn] file not found: C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\model_feature_engineered_datasets\df_model_line.csv
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv -> (33029, 30)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv -> (462, 8)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\bowler_stats.csv -> (318, 14)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial i

In [4]:
print("Similarity shape:", batsman_similarity.shape)
print("First 10 index names:", list(batsman_similarity.index[:10]))
print("First 10 column names:", list(batsman_similarity.columns[:10]))
print("Does 'A Balbirnie' exist as index?:", 'A Balbirnie' in batsman_similarity.index)
print("Does 'A Balbirnie' exist as column?:", 'A Balbirnie' in batsman_similarity.columns)


Similarity shape: (462, 462)
First 10 index names: ['Ab De Villiers', 'Aamir Kaleem', 'Aaron Finch', 'Aaron Johnson', 'Aaron Jones', 'Aaron Phangiso', 'Aasif Sheikh', 'Aayan Afzal Khan', 'Abinash Bohara', 'Adam Milne']
First 10 column names: ['Ab De Villiers', 'Aamir Kaleem', 'Aaron Finch', 'Aaron Johnson', 'Aaron Jones', 'Aaron Phangiso', 'Aasif Sheikh', 'Aayan Afzal Khan', 'Abinash Bohara', 'Adam Milne']
Does 'A Balbirnie' exist as index?: False
Does 'A Balbirnie' exist as column?: False
